# Ambara — Step-by-Step Colab Runner

Each pipeline step has its own cell. Run **Environment Setup** once, then execute
any step independently — no need to run everything in sequence.

**Workflow overview:**
1. Fill in the config cell and run **Environment Setup**
2. Pick any step cell, set its variables, and run it

| Step | What it does |
|------|--------------|
| 1 · Ingest | Download (if URL) + extract + sync to Supabase |
| 2 · Export | Export corrected clips as HuggingFace training dataset |
| 3 · Train | Fine-tune Whisper on the exported dataset |
| 4 · Redraft | Re-transcribe pending clips with the fine-tuned model |

- Supabase credentials come from `.env` stored in Google Drive (mounted once).
- Trained models can be pushed to HuggingFace Hub automatically.

In [ ]:
# ---- Google Drive (only for .env) ----
DRIVE_ROOT = "ambara"

# ---- Repository ----
REPO_URL = "https://github.com/ny-randriantsarafara/ny-feoko.git"
REPO_BRANCH = "main"

# ---- HuggingFace ----
HF_TOKEN = ""  # Write token — required for --push-to-hub

## Environment Setup

Run this cell once per session. Mounts Drive (for `.env`), clones/updates the repo,
and installs dependencies.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
DRIVE_BASE = DRIVE_MOUNT / "MyDrive" / DRIVE_ROOT
REPO_DIR = Path("/content/ny-feoko")

# ---- Mount Google Drive (for .env) ----
if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"Drive mounted at {DRIVE_MOUNT}")

# ---- Clone or update repo ----
if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
    print(f"Repo updated: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print(f"Repo cloned: {REPO_DIR}")

os.chdir(REPO_DIR)

# ---- Symlink .env from Drive (if present) ----
env_drive = DRIVE_BASE / ".env"
env_local = REPO_DIR / ".env"
if env_drive.exists():
    if env_local.is_symlink() or env_local.exists():
        env_local.unlink()
    env_local.symlink_to(env_drive)
    print(f"  .env -> {env_drive}")
else:
    print("WARNING: No .env found on Drive. Place it at My Drive/ambara/.env")

# ---- Python version check ----
v = sys.version_info
print(f"\nPython {v.major}.{v.minor}.{v.micro}")
if v < (3, 11):
    print("WARNING: This project requires Python >= 3.11.")

# ---- Install dependencies ----
subprocess.run(["make", "colab-install"], check=True, cwd=str(REPO_DIR))

# ---- HuggingFace login ----
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace Hub.")

# ---- Environment summary ----
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
print(f"\n{'=' * 50}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()} ({gpu_name})")
print(f"Repo:    {REPO_DIR}")
print(f"{'=' * 50}")
print("Setup complete.")

---
## Steps

Each cell below is independent. Set its variables and run it whenever you need that step.

## 1 · Ingest

Download audio (if URL), extract speech clips, and sync to Supabase — all in one command.

In [ ]:
import os
import subprocess
from pathlib import Path

INGEST_INPUT        = "https://youtube.com/watch?v=..."  # YouTube URL or local file path
INGEST_LABEL        = "my-recording"                      # Run label
INGEST_DEVICE       = "cuda"                              # cuda | mps | cpu
INGEST_WHISPER_MODEL = "small"                            # Whisper size (tiny/base/small/medium/large)
INGEST_WHISPER_HF   = ""                                  # HuggingFace model ID (overrides WHISPER_MODEL)
INGEST_VAD_THRESHOLD = 0.35                               # VAD speech probability threshold
INGEST_SPEECH_THRESHOLD = 0.35                            # Minimum speech classifier score

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "apps" / "api" / "src")

cmd = [
    "python", "-m", "ports.cli.app", "ingest",
    INGEST_INPUT,
    "--device", INGEST_DEVICE,
    "--whisper-model", INGEST_WHISPER_MODEL,
    "--vad-threshold", str(INGEST_VAD_THRESHOLD),
    "--speech-threshold", str(INGEST_SPEECH_THRESHOLD),
]
if INGEST_LABEL:
    cmd += ["--label", INGEST_LABEL]
if INGEST_WHISPER_HF:
    cmd += ["--whisper-hf", INGEST_WHISPER_HF]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True, env=env)

## 2 · Export Training Data

Download corrected clips from Supabase and build a HuggingFace-compatible training dataset.

In [ ]:
import os
import subprocess
from pathlib import Path

EXPORT_RUN_IDS   = []              # Run UUID(s) to export — get from web UI or Supabase
EXPORT_OUTPUT    = "data/output"   # Parent directory for the exported dataset
EXPORT_EVAL_SPLIT = 0.1            # Fraction reserved for evaluation (0.0 – 0.5)

if not EXPORT_RUN_IDS:
    raise ValueError("Set EXPORT_RUN_IDS to the run UUIDs from the web UI.")

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "apps" / "api" / "src")

cmd = [
    "python", "-m", "ports.cli.app", "export",
    "-o", EXPORT_OUTPUT,
    "--eval-split", str(EXPORT_EVAL_SPLIT),
]
for run_id in EXPORT_RUN_IDS:
    cmd += ["--run-id", run_id]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True, env=env)

## 3 · Train

Fine-tune Whisper on the exported training dataset.

In [ ]:
import os
import subprocess
from pathlib import Path

TRAIN_DATA_DIR    = "data/output"             # Path to exported training dataset
TRAIN_BASE_MODEL  = "openai/whisper-small"    # HuggingFace model to fine-tune
TRAIN_DEVICE      = "cuda"                    # cuda | mps | cpu | auto
TRAIN_EPOCHS      = 10
TRAIN_BATCH_SIZE  = 4
TRAIN_LR          = 1e-5
TRAIN_OUTPUT_DIR  = "models/whisper-mg-v1"    # Local save path
TRAIN_PUSH_TO_HUB = ""                        # HuggingFace repo ID (e.g. user/whisper-mg)

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "apps" / "api" / "src")

cmd = [
    "python", "-m", "ports.cli.app", "train",
    "-d", TRAIN_DATA_DIR,
    "-o", TRAIN_OUTPUT_DIR,
    "--device", TRAIN_DEVICE,
    "--base-model", TRAIN_BASE_MODEL,
    "--epochs", str(TRAIN_EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--lr", str(TRAIN_LR),
]
if TRAIN_PUSH_TO_HUB:
    cmd += ["--push-to-hub", TRAIN_PUSH_TO_HUB]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True, env=env)

## 4 · Redraft

Re-transcribe pending clips with the fine-tuned model and update Supabase drafts.

In [ ]:
import os
import subprocess
from pathlib import Path

REDRAFT_RUN_IDS = []                          # Run UUID(s) to redraft — same as export
REDRAFT_MODEL   = "models/whisper-mg-v1/model"  # Local model path or HuggingFace repo ID
REDRAFT_DEVICE  = "cuda"                      # cuda | mps | cpu | auto

if not REDRAFT_RUN_IDS:
    raise ValueError("Set REDRAFT_RUN_IDS to the run UUIDs from the web UI.")

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "apps" / "api" / "src")

cmd = [
    "python", "-m", "ports.cli.app", "redraft",
    "--model", REDRAFT_MODEL,
    "--device", REDRAFT_DEVICE,
]
for run_id in REDRAFT_RUN_IDS:
    cmd += ["--run-id", run_id]

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, check=True, env=env)